# LLM APIs and Embeddings

**CSCI 394 -- Spring 2026 Tutorial**

The previous tutorial used GPT-2 running *locally*. In practice, the most powerful LLMs
(Claude, GPT-4, Gemini) are accessed via **API** -- you send a request over the network
and receive a response. This tutorial covers:

## What you will learn

1. How to call a production LLM API (Anthropic Claude)
2. Streaming responses for real-time output
3. Multi-turn conversations and system prompts
4. Tool use: how LLMs call external functions
5. **Embeddings**: dense vector representations of text
6. Semantic similarity search
7. **Retrieval-Augmented Generation (RAG)**: connecting LLMs to external knowledge

## Prerequisites

```bash
pip install anthropic sentence-transformers faiss-cpu numpy matplotlib
```

You need an Anthropic API key set as an environment variable:
```bash
export ANTHROPIC_API_KEY='your-key-here'
```
Get one at https://console.anthropic.com (free tier available).

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install anthropic sentence-transformers faiss-cpu

import os
import time
import json
import numpy as np
import matplotlib.pyplot as plt

# Check for API key
api_key = os.environ.get("ANTHROPIC_API_KEY", "")
if api_key:
    print("ANTHROPIC_API_KEY found.")
else:
    print("WARNING: ANTHROPIC_API_KEY not set. Parts 1-4 require it.")
    print("Parts 5-7 (embeddings, RAG) work without an API key.")

---

## Part 1: Your First API Call

The simplest possible interaction: send a message, receive a reply.

Key difference from running a local model:
- The model runs on Anthropic's servers (hundreds of GPUs)
- You pay per token (input + output)
- You get a frontier model without managing hardware
- Latency is ~100-500 ms for the first token (network + prefill)

In [ ]:
import anthropic

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from environment

# The simplest API call
t0 = time.perf_counter()
message = client.messages.create(
    model="claude-haiku-4-5-20251001",  # fastest/cheapest Claude model
    max_tokens=200,
    messages=[
        {"role": "user", "content": "Explain what a Transformer is in 2 sentences."}
    ]
)
elapsed = time.perf_counter() - t0

response_text = message.content[0].text
print(response_text)
print()
print(f"Time to full response: {elapsed:.2f}s")
print(f"Input tokens:  {message.usage.input_tokens}")
print(f"Output tokens: {message.usage.output_tokens}")
print(f"Throughput:    {message.usage.output_tokens / elapsed:.1f} tokens/s")

---

## Part 2: System Prompts and Multi-Turn Conversations

A **system prompt** sets the model's persona and constraints *before* the user speaks.
A **multi-turn conversation** alternates `user` and `assistant` messages — the model
sees the full history at each step.

This is how chatbots like Claude.ai work under the hood.

In [ ]:
def chat(messages, system="", model="claude-haiku-4-5-20251001", max_tokens=300):
    """Send a conversation and return the assistant's reply."""
    kwargs = dict(model=model, max_tokens=max_tokens, messages=messages)
    if system:
        kwargs["system"] = system
    msg = client.messages.create(**kwargs)
    return msg.content[0].text


# System prompt sets persona
system = """You are an HPC tutor for undergraduates at Wheaton College.
Explain concepts clearly using analogies. Be concise -- 2-3 sentences per answer."""

# Build up a multi-turn conversation
conversation = []

questions = [
    "What is MPI and why do we need it?",
    "How is that different from OpenMP?",
    "Which should I use for training a neural network on a cluster?",
]

for q in questions:
    conversation.append({"role": "user", "content": q})
    reply = chat(conversation, system=system)
    conversation.append({"role": "assistant", "content": reply})
    print(f"Q: {q}")
    print(f"A: {reply}")
    print()

### Exercise 1: Role-play as an expert

Change the system prompt to make Claude play a different role. Try:
- A skeptical code reviewer who always points out edge cases
- A student who asks follow-up questions
- A physics professor explaining ML using physics analogies

Notice how the *same model* behaves very differently with different system prompts.

In [ ]:
# YOUR CODE HERE
my_system = "You are a skeptical code reviewer. Always point out at least one potential bug."

reply = chat(
    [{"role": "user", "content": "Here's my MPI code: for i in range(n): MPI.Send(data[i], dest=i)"}],
    system=my_system,
)
print(reply)

---

## Part 3: Streaming Responses

By default the API waits until the full response is ready before returning it.
**Streaming** delivers tokens as they are generated -- this is how Claude.ai
shows text appearing word-by-word.

For HPC applications: streaming reduces *time to first token* (TTFT) which matters
for interactive use, but total throughput (tokens/s) is similar.

In [ ]:
import sys

print("Streaming response (tokens appear as generated):")
print("-" * 60)

t0 = time.perf_counter()
first_token_time = None
token_count = 0

with client.messages.stream(
    model="claude-haiku-4-5-20251001",
    max_tokens=200,
    messages=[{
        "role": "user",
        "content": "List 5 key differences between data parallelism and model parallelism."
    }]
) as stream:
    for text_chunk in stream.text_stream:
        if first_token_time is None:
            first_token_time = time.perf_counter() - t0
        print(text_chunk, end="", flush=True)
        token_count += 1

total_time = time.perf_counter() - t0
print()
print("-" * 60)
print(f"Time to first token: {first_token_time:.3f}s")
print(f"Total time:          {total_time:.3f}s")

---

## Part 4: Tool Use (Function Calling)

Modern LLMs can call **tools** -- external functions you define. The model decides
*when* and *how* to call each tool based on the user's request.

This is powerful because it connects the LLM's language ability to real computation:
databases, calculators, simulators, or any Python function.

### How it works
1. You describe your tools as JSON schemas
2. The model outputs a structured tool call (name + arguments)
3. Your code executes the tool and returns the result
4. The model continues with the result in context

In [ ]:
# Define tools as JSON schemas
tools = [
    {
        "name": "compute_flops",
        "description": "Compute the training FLOPs for an LLM using the Chinchilla formula C = 6*N*D.",
        "input_schema": {
            "type": "object",
            "properties": {
                "params": {"type": "number", "description": "Number of model parameters"},
                "tokens": {"type": "number", "description": "Number of training tokens"},
            },
            "required": ["params", "tokens"],
        },
    },
    {
        "name": "compute_memory_gb",
        "description": "Compute GPU memory needed to store an LLM at a given precision.",
        "input_schema": {
            "type": "object",
            "properties": {
                "params": {"type": "number", "description": "Number of model parameters"},
                "precision": {"type": "string", "enum": ["fp32", "fp16", "int8", "int4"],
                              "description": "Weight precision"},
            },
            "required": ["params", "precision"],
        },
    },
]


def execute_tool(name, inputs):
    """Run the requested tool and return the result as a string."""
    if name == "compute_flops":
        C = 6 * inputs["params"] * inputs["tokens"]
        return f"{C:.2e} FLOPs ({C/1e18:.2f} ExaFLOPs)"
    elif name == "compute_memory_gb":
        bytes_per = {"fp32": 4, "fp16": 2, "int8": 1, "int4": 0.5}[inputs["precision"]]
        gb = inputs["params"] * bytes_per / 1e9
        return f"{gb:.1f} GB"
    return "Unknown tool"


def run_with_tools(user_message, tools, model="claude-haiku-4-5-20251001"):
    """Agentic loop: let the model call tools until it has an answer."""
    messages = [{"role": "user", "content": user_message}]
    
    while True:
        response = client.messages.create(
            model=model, max_tokens=500, tools=tools, messages=messages
        )
        
        # Collect any tool calls
        tool_calls = [b for b in response.content if b.type == "tool_use"]
        
        if not tool_calls:
            # No more tool calls -- return the final text
            return next(b.text for b in response.content if b.type == "text")
        
        # Execute each tool and build the result message
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for tc in tool_calls:
            result = execute_tool(tc.name, tc.input)
            print(f"  [tool] {tc.name}({tc.input}) -> {result}")
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": tc.id,
                "content": result,
            })
        messages.append({"role": "user", "content": tool_results})


# Ask a question that requires computation
question = (
    "Llama 3 405B has 405 billion parameters and was trained on 15 trillion tokens. "
    "How many ExaFLOPs did training require, and how much memory does it need in BF16?"
)
print(f"Question: {question}\n")
answer = run_with_tools(question, tools)
print(f"\nAnswer: {answer}")

### Exercise 2: Add a new tool

Add a `gpu_hours` tool that computes how many A100-hours a training run would take
given FLOPs and GPU efficiency. Test it by asking:
> "How many A100-hours would it take to train GPT-3?"

In [ ]:
# YOUR CODE HERE
# Add a gpu_hours tool to the tools list and execute_tool function

---

## Part 5: Text Embeddings

An **embedding** is a dense numeric vector that represents the *meaning* of text.
Two semantically similar sentences have embeddings that are close together in vector space.

Embeddings are the foundation of:
- Semantic search (find documents by meaning, not keywords)
- Clustering (group similar texts)
- Classification
- Retrieval-Augmented Generation (RAG)

We use `sentence-transformers` which provides open-weight embedding models
that run locally -- no API key needed.

In [ ]:
from sentence_transformers import SentenceTransformer

# all-MiniLM-L6-v2: 22M params, fast, good quality
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "MPI enables distributed memory parallelism across nodes.",
    "Message Passing Interface coordinates work across multiple machines.",
    "OpenMP uses shared memory threads within a single node.",
    "CUDA kernels run thousands of threads on a GPU.",
    "The cat sat on the mat.",
    "Transformers use self-attention to model token relationships.",
]

embeddings = embed_model.encode(sentences, normalize_embeddings=True)

print(f"Embedding shape: {embeddings.shape}")
print(f"Each sentence -> {embeddings.shape[1]}-dimensional vector")

In [ ]:
# Cosine similarity matrix (embeddings are normalized, so dot product = cosine similarity)
sim_matrix = embeddings @ embeddings.T

# Truncate labels for display
short_labels = [s[:35] + "..." if len(s) > 35 else s for s in sentences]

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(sim_matrix, cmap="RdYlGn", vmin=0, vmax=1)

ax.set_xticks(range(len(sentences)))
ax.set_yticks(range(len(sentences)))
ax.set_xticklabels(short_labels, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(short_labels, fontsize=8)
ax.set_title("Semantic Similarity (cosine)")

for i in range(len(sentences)):
    for j in range(len(sentences)):
        ax.text(j, i, f"{sim_matrix[i,j]:.2f}", ha="center", va="center", fontsize=8)

plt.colorbar(im)
plt.tight_layout()
plt.show()

print("Observation: the two MPI sentences should be highly similar (~0.8+).")
print("The cat sentence should be unrelated to everything (~0.1-0.2).")

### Dimensionality reduction: visualizing embeddings in 2D

We can project 384-dimensional embeddings to 2D using **PCA** to see
whether semantically similar sentences cluster together.

In [ ]:
from sklearn.decomposition import PCA

# More sentences to make the visualization interesting
corpus = [
    # MPI cluster
    "MPI_Send transmits data to another process.",
    "MPI_Recv waits for and receives a message.",
    "MPI_Allreduce sums values across all ranks.",
    "Distributed memory programming requires explicit communication.",
    # GPU cluster
    "CUDA threads execute in parallel on the GPU.",
    "cudaMemcpy transfers data between host and device.",
    "GPU warps execute 32 threads in lockstep.",
    "NVIDIA A100 has 80 GB of HBM2e memory.",
    # ML cluster
    "The Transformer uses multi-head self-attention.",
    "Backpropagation computes gradients for training.",
    "Batch normalization stabilizes deep network training.",
    "Stochastic gradient descent minimizes the loss.",
    # Unrelated
    "The weather today is sunny and warm.",
    "Pizza is a popular dish in Italy.",
]

colors = ["steelblue"]*4 + ["coral"]*4 + ["mediumseagreen"]*4 + ["gray"]*2
group_labels = ["MPI"]*4 + ["GPU"]*4 + ["ML"]*4 + ["Other"]*2

vecs = embed_model.encode(corpus, normalize_embeddings=True)
coords = PCA(n_components=2).fit_transform(vecs)

fig, ax = plt.subplots(figsize=(9, 7))
for i, (x, y) in enumerate(coords):
    ax.scatter(x, y, color=colors[i], s=80, zorder=3)
    ax.annotate(corpus[i][:40], (x, y), fontsize=7, xytext=(5, 3), textcoords="offset points")

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="steelblue", label="MPI"),
    Patch(facecolor="coral", label="GPU"),
    Patch(facecolor="mediumseagreen", label="ML"),
    Patch(facecolor="gray", label="Other"),
]
ax.legend(handles=legend_elements, loc="best")
ax.set_title("Sentence Embeddings Projected to 2D (PCA)")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---

## Part 6: Semantic Search with FAISS

**FAISS** (Facebook AI Similarity Search) is a library for fast nearest-neighbor
search in high-dimensional spaces. It's used at production scale by companies
storing billions of embeddings.

The idea: build an *index* of document embeddings, then query it with a question embedding
to find the most relevant documents in milliseconds -- even with millions of documents.

In [ ]:
import faiss

# Our "knowledge base" -- HPC course FAQ
documents = [
    "MPI (Message Passing Interface) is a standard for distributed memory parallel programming. Processes communicate by explicitly sending and receiving messages.",
    "OpenMP is an API for shared memory multiprocessing. It uses compiler directives (#pragma omp) to parallelize loops and sections.",
    "A GPU contains thousands of small cores optimized for throughput. CUDA is NVIDIA's programming model for GPUs.",
    "Data parallelism splits the dataset across workers. Each worker holds a full model copy and processes a different batch.",
    "Model parallelism splits the model itself across devices. Different layers or layer shards run on different GPUs.",
    "The KV cache stores key and value tensors from previous tokens during autoregressive generation, avoiding redundant computation.",
    "Quantization reduces model precision from FP32 to FP16, INT8, or INT4. This saves memory and often speeds up inference.",
    "LoRA (Low-Rank Adaptation) fine-tunes LLMs efficiently by adding small trainable matrices to frozen pretrained weights.",
    "Tensor parallelism splits individual weight matrices across GPUs. It requires high-bandwidth interconnects like NVLink.",
    "Flash Attention computes attention in tiles that fit in GPU SRAM, reducing HBM memory traffic and enabling longer context windows.",
    "The Transformer's self-attention has O(n^2) complexity in sequence length n, which is why long contexts are expensive.",
    "Gradient checkpointing trades compute for memory: discard intermediate activations during the forward pass and recompute them during backprop.",
]

# Encode and index
doc_embeddings = embed_model.encode(documents, normalize_embeddings=True).astype(np.float32)
dim = doc_embeddings.shape[1]

# IndexFlatIP: exact inner product search (equivalent to cosine with normalized vectors)
index = faiss.IndexFlatIP(dim)
index.add(doc_embeddings)

print(f"Indexed {index.ntotal} documents, dimension {dim}")


def search(query, top_k=3):
    q_vec = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = index.search(q_vec, top_k)
    print(f"Query: '{query}'")
    print(f"Top {top_k} results:")
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), 1):
        print(f"  {rank}. [score={score:.3f}] {documents[idx]}")
    print()


search("How do I save GPU memory when training a large model?")
search("What is the difference between MPI and OpenMP?")
search("Why is long-context inference slow?")

---

## Part 7: Retrieval-Augmented Generation (RAG)

**RAG** combines semantic search with an LLM:

```
User question
     |
     v
Embed question  -->  FAISS index  -->  Top-k relevant documents
                                              |
                                              v
                             LLM prompt = question + documents
                                              |
                                              v
                                      Grounded answer
```

### Why RAG?
- LLMs have a knowledge cutoff and can hallucinate facts
- RAG grounds answers in real documents you control
- Much cheaper than fine-tuning
- Documents can be updated without retraining the model

### HPC connection
At scale (millions of documents), the FAISS index must be sharded across
multiple machines -- the same distributed systems challenges we study in this course.

In [ ]:
def rag_answer(question, top_k=3, model="claude-haiku-4-5-20251001"):
    """Answer a question using retrieval-augmented generation."""
    # Step 1: retrieve relevant documents
    q_vec = embed_model.encode([question], normalize_embeddings=True).astype(np.float32)
    scores, indices = index.search(q_vec, top_k)
    
    retrieved = [
        f"[Doc {rank}] {documents[idx]}"
        for rank, idx in enumerate(indices[0], 1)
    ]
    context = "\n".join(retrieved)
    
    # Step 2: construct an augmented prompt
    prompt = f"""Answer the question using ONLY the information in the documents below.
If the documents do not contain enough information, say so.

Documents:
{context}

Question: {question}
Answer:"""
    
    # Step 3: call the LLM
    if api_key:
        response = client.messages.create(
            model=model,
            max_tokens=400,
            messages=[{"role": "user", "content": prompt}]
        )
        answer = response.content[0].text
    else:
        answer = "(API key not set -- showing retrieved context only)"
    
    return answer, retrieved


# Test RAG
questions = [
    "How can I reduce the memory needed for inference?",
    "What hardware is needed for tensor parallelism?",
    "How does attention complexity affect long sequences?",
]

for q in questions:
    print(f"Question: {q}")
    answer, retrieved = rag_answer(q)
    print(f"Retrieved docs:")
    for doc in retrieved:
        print(f"  - {doc[:80]}...")
    print(f"Answer: {answer}")
    print("=" * 70)
    print()

### Exercise 3: Hallucination detection

Ask a question about something NOT in the knowledge base (e.g., about a topic
outside HPC). Compare:
1. What RAG returns (should say "not in documents")
2. What you get with a *direct* LLM call (may hallucinate details)

This demonstrates why RAG is safer for factual applications.

In [ ]:
# YOUR CODE HERE
# Try: "What is the boiling point of water?" or "Who won the 2024 US election?"

out_of_scope = "What is the population of Chicago?"

# RAG answer
answer, docs = rag_answer(out_of_scope)
print("RAG answer:")
print(answer)
print()

# Direct LLM answer (if API key available)
if api_key:
    direct = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=100,
        messages=[{"role": "user", "content": out_of_scope}]
    )
    print("Direct LLM answer (no retrieval):")
    print(direct.content[0].text)

---

## Part 8: Prompt Caching (HPC Cost Optimization)

When the same long context (system prompt, documents) is sent in many requests,
Anthropic's **prompt caching** reuses the computed KV representations.

This is the *same optimization* as the KV cache inside the model -- but at
the API level, persisted between separate requests.

**Cost savings**: cached input tokens cost 90% less than regular input tokens.
For RAG with a large document corpus, this can be the difference between
a practical and impractical system.

> **Note**: Prompt caching is a Claude-specific API feature. The cache
> breakpoint (`cache_control`) tells Anthropic to cache everything up to that point.

In [ ]:
if not api_key:
    print("Skipping: ANTHROPIC_API_KEY not set")
else:
    # A long system prompt that will be reused across many requests
    large_context = "\n".join([
        f"Document {i+1}: {doc}" for i, doc in enumerate(documents)
    ])

    def call_with_cache(question):
        """Use prompt caching for the document context."""
        return client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=200,
            system=[
                {
                    "type": "text",
                    "text": f"You are an HPC tutor. Reference these documents:\n{large_context}",
                    "cache_control": {"type": "ephemeral"},  # cache this block
                }
            ],
            messages=[{"role": "user", "content": question}],
        )

    questions_to_ask = [
        "What is LoRA?",
        "Explain Flash Attention.",
        "What is gradient checkpointing?",
    ]

    print(f"{'Request':>8s}  {'Input tokens':>14s}  {'Cache hit':>12s}  {'Time (s)':>10s}")
    print("-" * 52)

    for i, q in enumerate(questions_to_ask, 1):
        t0 = time.perf_counter()
        resp = call_with_cache(q)
        elapsed = time.perf_counter() - t0

        usage = resp.usage
        cache_read = getattr(usage, "cache_read_input_tokens", 0) or 0
        cache_creation = getattr(usage, "cache_creation_input_tokens", 0) or 0
        hit = f"{cache_read:,}" if cache_read else f"(creating: {cache_creation:,})"

        print(f"{i:>8d}  {usage.input_tokens:>14,}  {hit:>12s}  {elapsed:>10.2f}")

    print()
    print("Cache hit tokens are 90% cheaper than regular input tokens.")
    print("This matters a lot when the same large context is used for many queries.")

---

## Summary

| Concept | Key takeaway |
| ------- | ------------ |
| **API vs local** | Production LLMs run on provider infrastructure; you pay per token |
| **System prompts** | Set model persona/constraints; same model behaves very differently |
| **Streaming** | Delivers tokens as generated; reduces time-to-first-token |
| **Tool use** | LLMs can call structured external functions; enables agents |
| **Embeddings** | Dense vectors encoding semantic meaning; cosine similarity measures relatedness |
| **FAISS** | Fast nearest-neighbor search for embedding vectors at scale |
| **RAG** | Combine retrieval + generation for grounded, factual answers |
| **Prompt caching** | Reuse KV representations of repeated context; 90% cheaper |

### HPC connections

- Embedding databases at scale require distributed FAISS across many nodes
- Serving LLMs at high throughput requires batching, tensor parallelism, and quantization
- Prompt caching is the same idea as KV caching, applied across API requests
- Tool-use agents can trigger HPC jobs (submit PBS scripts, read job output, iterate)

### Further reading

- [Anthropic API docs](https://docs.anthropic.com)
- Lewis et al., "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks" (2020)
- Johnson et al., "Billion-scale similarity search with FAISS" (2017)
- Hu et al., "LoRA: Low-Rank Adaptation of Large Language Models" (2021)